# NB01 — Raw Dataset Audit + EDA + Data-Quality Verification

**Project:** Deep-learning flood mapping (Sen1Floods11 + Copernicus DEM + NASA IMERG)

**Purpose (ONLY):** audit the *raw* dataset as delivered on disk. Exploratory analysis + quality verification.

**This notebook does NOT:**
- preprocess / clean / resample / normalize / augment data
- create model-ready tensors (no `.npz`, no saved arrays)
- train models
- create new train/val/test splits (uses official Sen1Floods11 splits only)
- download anything or modify/overwrite raw files
- use Bolivia for training (Bolivia is a holdout)

**Design:** metadata-first, incremental statistics, memory-efficient. Safe to run top-to-bottom before NB02.

Expected ground truth to *verify* (not assume): 446 hand-labeled S1 chips — Train 252 / Val 89 / Test 90 / Bolivia 15.

## 1. Project configuration
All paths defined once here. No hard-coded paths elsewhere.

In [ ]:
from pathlib import Path

# --- Project / workspace ---
PROJECT_ROOT = Path(r"C:/Users/Swarnim/Desktop/ML projects/tsao")
NOTEBOOK_NAME = "NB01_Dataset_Audit"
RANDOM_SEED = 42

# --- Sen1Floods11 raw roots ---
SEN1_ROOT  = Path(r"C:/FloodProject/Sen1Floods11/v1.1")
S1_DIR     = SEN1_ROOT / "data" / "flood_events" / "HandLabeled" / "S1Hand"
LABEL_DIR  = SEN1_ROOT / "data" / "flood_events" / "HandLabeled" / "LabelHand"
SPLIT_DIR  = SEN1_ROOT / "splits" / "flood_handlabeled"

# --- Auxiliary modalities ---
DEM_DIR   = Path(r"C:/FloodProject/DEM")
IMERG_DIR = Path(r"C:/FloodProject/IMERG/data")

# --- Official split CSVs (no header in file; col0=S1Hand, col1=LabelHand) ---
SPLIT_FILES = {
    "train":    SPLIT_DIR / "flood_train_data.csv",
    "valid":    SPLIT_DIR / "flood_valid_data.csv",
    "test":     SPLIT_DIR / "flood_test_data.csv",
    "bolivia":  SPLIT_DIR / "flood_bolivia_data.csv",
}
EXPECTED_SPLITS = {"train": 252, "valid": 89, "test": 90, "bolivia": 15}
EXPECTED_TOTAL = 446

# --- Audit parameters (memory-efficient) ---
S1_STATS_SAMPLE_N = 30      # evenly-spaced subset for VV/VH value stats
DEM_MINMAX_SAMPLE_N = 5    # subset of DEM tiles for approx elevation range
DEM_SUBSAMPLE_STEP = 10    # stride when sampling DEM pixels for min/max
IMERG_STATS_SAMPLE_N = 2   # files to compute precip min/max/mean on
N_DISPLAY_SAMPLES = 6      # representative quick-look chips

for p in [S1_DIR, LABEL_DIR, SPLIT_DIR, DEM_DIR, IMERG_DIR]:
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")

## 2. Imports
Minimal stack only. No seaborn.

In [ ]:
import json
import os
import re
import collections

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import rasterio
import xarray as xr

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", matplotlib.__version__)
print("rasterio", rasterio.__version__)
print("xarray", xr.__version__)

rng = np.random.default_rng(RANDOM_SEED)
%matplotlib inline

## 3. Dataset inventory
Count files on disk. For DEM count **only** main files matching `Copernicus_DSM_COG_10_*_DEM.tif` (recursive — tiles live in subfolders). For IMERG count `*.nc4`.

In [ ]:
def inventory_summary():
    """Count raw files per modality (metadata only, no raster reads)."""
    s1_files = sorted(S1_DIR.glob("*.tif")) if S1_DIR.exists() else []
    lb_files = sorted(LABEL_DIR.glob("*.tif")) if LABEL_DIR.exists() else []
    # Recursive: DEM tiles are stored one-folder-per-tile.
    dem_files = sorted(DEM_DIR.rglob("Copernicus_DSM_COG_10_*_DEM.tif")) if DEM_DIR.exists() else []
    imerg_files = sorted(IMERG_DIR.glob("*.nc4")) if IMERG_DIR.exists() else []
    return s1_files, lb_files, dem_files, imerg_files

s1_files, lb_files, dem_files, imerg_files = inventory_summary()

inv = pd.DataFrame([
    {"modality": "Sentinel-1 (S1Hand)", "pattern": "*.tif (flat)",
     "count": len(s1_files), "example": s1_files[0].name if s1_files else None},
    {"modality": "Flood label (LabelHand)", "pattern": "*.tif (flat)",
     "count": len(lb_files), "example": lb_files[0].name if lb_files else None},
    {"modality": "Copernicus DEM (main only)", "pattern": "Copernicus_DSM_COG_10_*_DEM.tif (recursive)",
     "count": len(dem_files), "example": dem_files[0].name if dem_files else None},
    {"modality": "NASA IMERG (daily)", "pattern": "*.nc4",
     "count": len(imerg_files), "example": imerg_files[0].name if imerg_files else None},
])
display(inv)

print(f"S1 extensions: {sorted({p.suffix for p in s1_files})}")
print(f"Label extensions: {sorted({p.suffix for p in lb_files})}")
print("S1 filename examples:", [p.name for p in s1_files[:3]])
print("Label filename examples:", [p.name for p in lb_files[:3]])
print("DEM example paths (parent folders):", [str(p.parent.name) for p in dem_files[:3]])
print("IMERG examples:", [p.name for p in imerg_files[:3]])
print(f"\nExpected: S1=446, labels=446, IMERG=36 | Found: {len(s1_files)}, {len(lb_files)}, {len(imerg_files)}")

## 4. Official split audit
The CSVs have **no header**: each line is `S1Hand_filename,LabelHand_filename`. We inspect raw lines first, then parse with `header=None` — never with a default header assumption.

In [ ]:
# --- 4a. Inspect actual CSV structure (raw lines, no parsing assumptions) ---
for split, csv_path in SPLIT_FILES.items():
    print(f"--- {split}: {csv_path.name} ---")
    with open(csv_path, "r") as f:
        lines = [ln.rstrip("\n") for ln in f if ln.strip()]
    print(f"  n_lines (non-empty): {len(lines)}")
    for ln in lines[:3]:
        print(f"  RAW: {ln!r}  -> n_commas={ln.count(',')}")

# --- 4b. Parse correctly: header=None, two filename columns ---
split_s1, split_lb, split_df = {}, {}, {}
for split, csv_path in SPLIT_FILES.items():
    df = pd.read_csv(csv_path, header=None, names=["s1_file", "label_file"], dtype=str)
    df["s1_file"] = df["s1_file"].str.strip()
    df["label_file"] = df["label_file"].str.strip()
    split_df[split] = df
    split_s1[split] = df["s1_file"].tolist()
    split_lb[split] = df["label_file"].tolist()
    assert (df["s1_file"].str.endswith("_S1Hand.tif")).all(), f"{split}: unexpected S1 col values"
    assert (df["label_file"].str.endswith("_LabelHand.tif")).all(), f"{split}: unexpected label col values"

counts = {k: len(v) for k, v in split_s1.items()}
total_ref = sum(counts.values())
print("\nSplit counts:", counts, " Total:", total_ref)
print("Expected:  ", EXPECTED_SPLITS, " Total:", EXPECTED_TOTAL)

In [ ]:
# --- 4c. Integrity checks: duplicates, overlap, existence, 1:1 S1<->label ---
on_disk_s1 = {p.name for p in s1_files}
on_disk_lb = {p.name for p in lb_files}
results = []

def record(name, ok, detail=""):
    results.append((name, "PASS" if ok else "FAIL", detail))

# within-split duplicates
for split, names in split_s1.items():
    record(f"no duplicate S1 within '{split}'", len(set(names)) == len(names),
           f"{len(names)} rows, {len(set(names))} unique")

# between-split overlap
sets = {k: set(v) for k, v in split_s1.items()}
keys = list(sets)
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        ov = sets[keys[i]] & sets[keys[j]]
        record(f"no overlap {keys[i]} x {keys[j]}", len(ov) == 0, f"overlap={len(ov)}" + (f" e.g. {sorted(ov)[:2]}" if ov else ""))

# every referenced S1 exists; every S1 chip referenced exactly once
all_ref = [n for v in split_s1.values() for n in v]
missing = [n for n in all_ref if n not in on_disk_s1]
record("every referenced S1 exists on disk", len(missing) == 0, f"missing={len(missing)}" + (f" e.g. {missing[:3]}" if missing else ""))
unref = sorted(on_disk_s1 - set(all_ref))
record("every S1 chip referenced exactly once", len(unref) == 0 and len(all_ref) == len(set(all_ref)),
       f"unreferenced={len(unref)}" + (f" e.g. {unref[:3]}" if unref else ""))

# every S1 has matching LabelHand (both CSV reference and file on disk)
bad_map = []
for split in split_df:
    df = split_df[split]
    for s1n, lbn in zip(df["s1_file"], df["label_file"]):
        if lbn not in on_disk_lb or lbn.replace("_LabelHand.tif", "") != s1n.replace("_S1Hand.tif", ""):
            bad_map.append((split, s1n, lbn))
record("every S1 has matching LabelHand file", len(bad_map) == 0, f"mismatches={len(bad_map)}" + (f" e.g. {bad_map[:2]}" if bad_map else ""))

# expected counts
for split, exp in EXPECTED_SPLITS.items():
    record(f"count {split}=={exp}", counts.get(split) == exp, f"found={counts.get(split)}")
record("total==446", total_ref == EXPECTED_TOTAL, f"found={total_ref}")

audit_df = pd.DataFrame(results, columns=["check", "status", "detail"])
display(audit_df)
print("\nSPLIT AUDIT:", "ALL PASS" if (audit_df.status == "PASS").all() else "FAILURES PRESENT — see table")

## 5. Sentinel-1 inspection
Metadata pass over **all** chips (cheap) + numerical stats on an evenly-spaced subset (memory-efficient). NaNs are reported, never replaced.

In [ ]:
# --- 5a. Metadata sweep over all 446 chips ---
meta = []  # (name, count, h, w, dtype, crs_str, res_x, res_y, transform_tuple, bounds_tuple)
unreadable = []
for p in s1_files:
    try:
        with rasterio.open(p) as src:
            meta.append((p.name, src.count, src.height, src.width, tuple(str(d) for d in src.dtypes),
                           str(src.crs), round(float(src.res[0]), 12), round(float(src.res[1]), 12),
                           tuple(src.transform), tuple(src.bounds)))
    except Exception as e:
        unreadable.append((p.name, repr(e)))

mdf = pd.DataFrame(meta, columns=["file", "bands", "h", "w", "dtype", "crs", "res_x", "res_y", "transform", "bounds"])
print(f"readable: {len(mdf)}/{len(s1_files)}  unreadable: {len(unreadable)}")
if unreadable:
    print(unreadable[:5])
print("\nUnique band counts:", mdf["bands"].unique())
print("Unique shapes (h,w):", mdf[["h", "w"]].drop_duplicates().values.tolist())
print("Unique dtypes:", mdf["dtype"].unique())
print("Unique CRS:", mdf["crs"].unique())
print("Unique resolutions:", mdf[["res_x", "res_y"]].drop_duplicates().values.tolist())
print("Unique transforms:", mdf["transform"].nunique(), " Unique bounds:", mdf["bounds"].nunique(), "(bounds/transforms vary per chip — expected)")
display(mdf.head(3))

In [ ]:
# --- 5b. Numerical stats on an evenly-spaced subset (incremental, NaN-aware) ---
idx = np.linspace(0, len(s1_files) - 1, num=min(S1_STATS_SAMPLE_N, len(s1_files)), dtype=int)
sample_paths = [s1_files[i] for i in idx]
print(f"Sampling {len(sample_paths)} chips evenly across sorted file list (seed-free, deterministic).")

agg = {b: {"n": 0, "nan": 0, "sum": 0.0, "sumsq": 0.0, "min": np.inf, "max": -np.inf} for b in ("VV", "VH")}
for p in sample_paths:
    with rasterio.open(p) as src:
        arr = src.read()  # (2, 512, 512) float32 — one chip at a time
        assert arr.shape == (2, 512, 512), f"{p.name}: unexpected shape {arr.shape}"
    for bi, b in enumerate(("VV", "VH")):
        v = arr[bi].astype(np.float64)
        valid = v[~np.isnan(v)]
        a = agg[b]
        a["n"] += valid.size
        a["nan"] += int(np.isnan(v).sum())
        if valid.size:
            a["sum"] += float(valid.sum())
            a["sumsq"] += float((valid ** 2).sum())
            a["min"] = min(a["min"], float(valid.min()))
            a["max"] = max(a["max"], float(valid.max()))
    del arr

rows = []
for b in ("VV", "VH"):
    a = agg[b]
    mean = a["sum"] / a["n"] if a["n"] else np.nan
    var = a["sumsq"] / a["n"] - mean ** 2 if a["n"] else np.nan
    tot = a["n"] + a["nan"]
    rows.append({"band": b, "min_dB": round(a["min"], 3), "max_dB": round(a["max"], 3),
                 "mean_dB": round(mean, 3), "std_dB": round(float(np.sqrt(max(var, 0))), 3),
                 "nan_pct": round(100 * a["nan"] / tot, 4) if tot else np.nan,
                 "pixels_sampled": tot})
s1_stats = pd.DataFrame(rows)
display(s1_stats)
print("Values are Sentinel-1 SAR backscatter in dB (float32). NoData = NaN (reported, not replaced).")

## 6. Flood-label inspection
Valid values are only `-1` (NoData), `0` (non-flood), `1` (flood). Counts are accumulated chip-by-chip (never all in RAM). `-1` is reported separately — never as non-flood.

In [ ]:
# --- 6a. Metadata sweep + incremental pixel census over ALL 446 label chips ---
tot_nodata = tot_non = tot_flood = tot_other = 0
other_vals = collections.Counter()
per_chip_flood_pct = {}  # stem -> flood % among valid pixels
label_meta = []
bad_reads = []
for p in lb_files:
    try:
        with rasterio.open(p) as src:
            label_meta.append((p.name, src.count, src.height, src.width, str(src.dtypes[0]), str(src.crs)))
            lab = src.read(1)  # (512, 512), one chip at a time
    except Exception as e:
        bad_reads.append((p.name, repr(e)))
        continue
    u, c = np.unique(lab, return_counts=True)
    d = dict(zip(u.tolist(), (int(x) for x in c)))
    tot_nodata += d.get(-1, 0)
    tot_non    += d.get(0, 0)
    tot_flood  += d.get(1, 0)
    for k, v in d.items():
        if k not in (-1, 0, 1):
            tot_other += v
            other_vals[k] += v
    valid = int(lab.size - d.get(-1, 0))
    per_chip_flood_pct[p.name.replace("_LabelHand.tif", "_S1Hand.tif")] = (100.0 * d.get(1, 0) / valid) if valid else np.nan
    del lab

ldf = pd.DataFrame(label_meta, columns=["file", "bands", "h", "w", "dtype", "crs"])
print(f"readable: {len(ldf)}/{len(lb_files)}  unreadable: {len(bad_reads)}")
print("Unique band counts:", ldf["bands"].unique())
print("Unique shapes:", ldf[["h", "w"]].drop_duplicates().values.tolist())
print("Unique dtypes:", ldf["dtype"].unique())
print("Unique CRS:", ldf["crs"].unique())

total_px = tot_nodata + tot_non + tot_flood + tot_other
valid_px = tot_non + tot_flood
label_totals = pd.DataFrame([
    {"class": "-1 NoData",    "pixels": tot_nodata, "pct_of_all": round(100 * tot_nodata / total_px, 4)},
    {"class": "0 non-flood",  "pixels": tot_non,    "pct_of_all": round(100 * tot_non / total_px, 4)},
    {"class": "1 flood",      "pixels": tot_flood,  "pct_of_all": round(100 * tot_flood / total_px, 4)},
    {"class": "other (invalid)", "pixels": tot_other, "pct_of_all": round(100 * tot_other / total_px, 6)},
    {"class": "ALL pixels",   "pixels": total_px,   "pct_of_all": 100.0},
])
display(label_totals)
print(f"Flood % among VALID pixels (excl. NoData): {100 * tot_flood / valid_px:.4f}%  (valid={valid_px:,}, nodata={tot_nodata:,})")
print(f"Unexpected values: {dict(other_vals) if other_vals else 'none'}")

In [ ]:
# --- 6b. Per-chip flood-percentage distribution (446 chips) ---
chip_pct = pd.Series(per_chip_flood_pct, name="flood_pct_valid")
desc = chip_pct.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
display(desc.to_frame().T.round(4))
print(f"chips with 0% flood (valid pixels, no flood): {(chip_pct == 0).sum()}")
print(f"chips with 100% flood: {(chip_pct == 100).sum()}")
print(f"chips with all-NoData (NaN pct): {int(chip_pct.isna().sum())}")
print("\nMost flood-dominant chips:")
display(chip_pct.sort_values(ascending=False).head(5).to_frame())
print("Least flood-present (non-zero) chips:")
display(chip_pct[chip_pct > 0].sort_values().head(5).to_frame())

## 7. DEM inventory
Main DEM files only (`Copernicus_DSM_COG_10_*_DEM.tif`, recursive). Metadata sweep + readability check + approximate elevation range from a strided sample. No mosaicking/resampling here (NB02 work).

In [ ]:
# --- 7a. Metadata sweep over all main DEM tiles ---
dem_meta, dem_bad = [], []
for p in dem_files:
    try:
        with rasterio.open(p) as src:
            dem_meta.append((str(p.parent.name), p.name, src.count, src.width, src.height,
                               str(src.dtypes[0]), str(src.crs), round(float(src.res[0]), 12), round(float(src.res[1]), 12)))
    except Exception as e:
        dem_bad.append((str(p), repr(e)))

ddf = pd.DataFrame(dem_meta, columns=["tile_dir", "file", "bands", "w", "h", "dtype", "crs", "res_x", "res_y"])
print(f"main DEM tiles found: {len(ddf)} | readable: {len(ddf)} | unreadable: {len(dem_bad)}")
if dem_bad:
    print(dem_bad[:5])
print("\nCRS distribution:")
display(ddf["crs"].value_counts().to_frame("n_tiles"))
print("Resolution distribution:")
display(ddf.groupby(["res_x", "res_y"]).size().to_frame("n_tiles"))
print("Dtype distribution:")
display(ddf["dtype"].value_counts().to_frame("n_tiles"))
print("Tile size distribution (w,h):")
display(ddf.groupby(["w", "h"]).size().to_frame("n_tiles"))

# --- 7b. Approximate elevation min/max from a small strided sample ---
k = min(DEM_MINMAX_SAMPLE_N, len(dem_files))
step_idx = np.linspace(0, len(dem_files) - 1, num=k, dtype=int) if dem_files else []
gmin, gmax = np.inf, -np.inf
for i in step_idx:
    p = dem_files[int(i)]
    with rasterio.open(p) as src:
        samp = src.read(1, out_shape=(src.height // DEM_SUBSAMPLE_STEP, src.width // DEM_SUBSAMPLE_STEP),
                          resampling=rasterio.enums.Resampling.nearest)
        v = samp[np.isfinite(samp)]
        if v.size:
            gmin, gmax = min(gmin, float(v.min())), max(gmax, float(v.max()))
        print(f"  {p.parent.name}: shape={src.shape} sampled_valid={v.size:,}")
        del samp, v
print(f"\nApprox elevation range over {k} sampled tiles (stride-1/{DEM_SUBSAMPLE_STEP} nearest): min={gmin:.2f} m, max={gmax:.2f} m")
print("All tiles readable:", "YES" if not dem_bad else f"NO ({len(dem_bad)} failures)")

## 8. IMERG inventory
36 daily NetCDF4 files. Variable of interest: `precipitation`. Per-file header inspection (no bulk loads); value stats on 1–2 files only.

In [ ]:
# --- 8a. Per-file header audit ---
date_re = re.compile(r"(\d{8})")
rows, seen_dates, files_missing_precip, open_errors = [], collections.Counter(), [], []
for p in imerg_files:
    m = date_re.search(p.name)
    datestr = m.group(1) if m else "UNPARSED"
    seen_dates[datestr] += 1
    try:
        with xr.open_dataset(p) as ds:
            has_precip = "precipitation" in ds.data_vars
            if not has_precip:
                files_missing_precip.append(p.name)
            dims = dict(ds.sizes)
            coords = {c: (str(ds[c].values.min()), str(ds[c].values.max()), str(ds[c].values.dtype)) for c in ("lon", "lat", "time") if c in ds.coords}
            rows.append({"file": p.name, "date": datestr, "has_precipitation": has_precip,
                         "dim_time": dims.get("time"), "dim_lat": dims.get("lat"), "dim_lon": dims.get("lon"),
                         "lon_min": coords.get("lon", (None,))[0], "lon_max": coords.get("lon", (None,))[1] if "lon" in coords else None,
                         "lat_min": coords.get("lat", (None,))[0], "lat_max": coords.get("lat", (None,))[1] if "lat" in coords else None})
    except Exception as e:
        open_errors.append((p.name, repr(e)))

idf = pd.DataFrame(rows).sort_values("date").reset_index(drop=True)
display(idf[["date", "file", "has_precipitation", "dim_time", "dim_lat", "dim_lon"]].head(10))
print(f"\nFiles: {len(idf)} (expected 36) | unique dates: {idf['date'].nunique()} | all have precipitation: {(idf['has_precipitation']).all()}")
print("Duplicate dates:", {d: c for d, c in seen_dates.items() if c > 1} or "none")
print("Dims distribution (time,lat,lon):", idf[["dim_time", "dim_lat", "dim_lon"]].drop_duplicates().values.tolist(), "(expected [1, 1800, 3600] = 0.1-deg global grid)")
print("Lon range:", idf['lon_min'].iloc[0], "→", idf['lon_max'].iloc[0])
print("Lat range:", idf['lat_min'].iloc[0], "→", idf['lat_max'].iloc[0])
if open_errors:
    print("OPEN ERRORS:", open_errors[:3])
if files_missing_precip:
    print("MISSING precipitation:", files_missing_precip)

In [ ]:
# --- 8b. Coordinate ordering + value sample on 1-2 files ---
with xr.open_dataset(str(IMERG_DIR / idf['file'].iloc[0])) as ds0:
    lon = ds0["lon"].values
    lat = ds0["lat"].values
    print(f"lon monotonic increasing: {bool(np.all(np.diff(lon) > 0))}  n={lon.size} step~{np.diff(lon).mean():.4f}")
    print(f"lat monotonic decreasing: {bool(np.all(np.diff(lat) < 0))}  n={lat.size} (descending is the IMERG convention)")
    print(f"precipitation dtype={ds0['precipitation'].dtype} dims={dict(ds0['precipitation'].sizes)}")
    print(f"precipitation attrs: {dict(ds0['precipitation'].attrs)}")

for fname in idf["file"].head(IMERG_STATS_SAMPLE_N):
    with xr.open_dataset(str(IMERG_DIR / fname)) as ds:
        pr = ds["precipitation"].isel(time=0).values  # (lat, lon); one file at a time
        print(f"\n{fname}: shape={pr.shape} dtype={pr.dtype}")
        print(f"  min={np.nanmin(pr):.4f} max={np.nanmax(pr):.4f} mean={np.nanmean(pr):.6f} NaN={int(np.isnan(pr).sum()):,}/{pr.size:,}")
        del pr
print("\nNo normalization/resampling applied — raw daily precipitation (mm/day scale per IMERG docs).")

## 9. Geographic / event distribution
Event/country prefix comes from the S1 filename stem (`<Event>_<chipid>_S1Hand.tif`) — the same prefix used by Sen1Floods11 for its 11 flood events. Cross-tabulated against the official splits (no invented geography).

In [ ]:
def event_of(s1_name: str) -> str:
    """Event/country prefix of a Sen1Floods11 HandLabeled chip filename."""
    return s1_name.split("_")[0]

# chip -> split lookup from official CSVs
chip_to_split = {}
for split, names in split_s1.items():
    for n in names:
        chip_to_split[n] = split

geo_rows = []
for p in s1_files:
    geo_rows.append({"event": event_of(p.name), "split": chip_to_split.get(p.name, "UNREFERENCED")})
gdf = pd.DataFrame(geo_rows)
ct = pd.crosstab(gdf["event"], gdf["split"])
for c in ["train", "valid", "test", "bolivia"]:
    if c not in ct.columns:
        ct[c] = 0
ct = ct[["train", "valid", "test", "bolivia"]]
ct["total"] = ct.sum(axis=1)
ct = ct.sort_values("total", ascending=False)
ct.loc["TOTAL"] = ct.sum(axis=0)
display(ct)
print(f"\nDistinct events: {gdf['event'].nunique()} ({sorted(gdf['event'].unique())})")
print(f"Unreferenced chips (should be 0): {(gdf['split'] == 'UNREFERENCED').sum()}")

## 10. Sample visualizations
Representative quick-looks (VV / VH / label) from distinct events × splits, plus VV/VH pixel histograms and per-chip flood-% distribution. Raw values only.

In [ ]:
# --- 10a. Deterministic representative selection: distinct (split, event) coverage ---
by_split_event = collections.defaultdict(list)
for split, names in split_s1.items():
    for n in sorted(names):
        by_split_event[(split, event_of(n))].append(n)

wanted = [("train", None), ("valid", None), ("test", None), ("bolivia", None), (None, None), (None, None)]
reps, used_events = [], set()
for s, _ in wanted:
    cands = [(k, v[0]) for k, v in sorted(by_split_event.items()) if (s is None or k[0] == s) and k[1] not in used_events and v]
    if not cands:
        cands = [(k, v[0]) for k, v in sorted(by_split_event.items()) if (s is None or k[0] == s) and v]
    k, fname = cands[0]
    reps.append((k[0], k[1], fname))
    used_events.add(k[1])
reps = reps[:N_DISPLAY_SAMPLES]
for s, e, f in reps:
    print(f"  split={s:7s} event={e:10s} file={f}  flood%={per_chip_flood_pct.get(f, float('nan')):.2f}")

In [ ]:
# --- 10b. Quick-look panels: VV | VH | label (-1/0/1 distinguished) ---
label_cmap = ListedColormap(["#808080", "#1a1a1a", "#00b3ff"])  # -1 grey, 0 near-black, 1 cyan-blue

def robust_limits(a, lo=2, hi=98):
    v = a[np.isfinite(a)]
    return (float(np.percentile(v, lo)), float(np.percentile(v, hi))) if v.size else (0, 1)

fig, axes = plt.subplots(len(reps), 3, figsize=(9, 3 * len(reps)))
for i, (split, event, fname) in enumerate(reps):
    with rasterio.open(str(S1_DIR / fname)) as src:
        s1 = src.read()
    with rasterio.open(str(LABEL_DIR / fname.replace("_S1Hand.tif", "_LabelHand.tif"))) as src:
        lab = src.read(1)
    vv, vh = s1[0], s1[1]
    vmin_vv, vmax_vv = robust_limits(vv)
    vmin_vh, vmax_vh = robust_limits(vh)
    ax = axes[i]
    ax[0].imshow(vv, cmap="gray", vmin=vmin_vv, vmax=vmax_vv)
    ax[0].set_title(f"{event} | {split}\nVV dB [{vmin_vv:.1f},{vmax_vv:.1f}] p2-98")
    ax[1].imshow(vh, cmap="gray", vmin=vmin_vh, vmax=vmax_vh)
    ax[1].set_title(f"VH dB [{vmin_vh:.1f},{vmax_vh:.1f}] p2-98\n{fname}")
    im = ax[2].imshow(lab, cmap=label_cmap, vmin=-1, vmax=1, interpolation="nearest")
    fpct = per_chip_flood_pct.get(fname, float("nan"))
    ax[2].set_title(f"Label (-1 grey|0 blk|1 blue) flood={fpct:.1f}%")
    for a in ax:
        a.set_xticks([]); a.set_yticks([])
    del s1, lab, vv, vh
fig.suptitle("Representative raw chips — VV / VH (dB, p2–98 stretch for display only) / Label", y=0.995)
fig.tight_layout()
plt.show()

In [ ]:
# --- 10c. Distributions: VV/VH pixel values + per-chip flood % ---
# Subsample pixels from the S1 stats subset (stride) to keep the hist cheap.
vv_pool, vh_pool = [], []
for p in sample_paths[::3]:
    with rasterio.open(p) as src:
        a = src.read()
    vv_pool.append(a[0][::4, ::4].ravel())
    vh_pool.append(a[1][::4, ::4].ravel())
    del a
vv_all = np.concatenate(vv_pool); vh_all = np.concatenate(vh_pool)
vv_all = vv_all[np.isfinite(vv_all)]; vh_all = vh_all[np.isfinite(vh_all)]
print(f"hist pixels: VV={vv_all.size:,}, VH={vh_all.size:,} (finite only, strided sample)")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].hist(vv_all, bins=80)
axes[0].set_title("VV (dB) — sampled pixels")
axes[0].set_xlabel("dB"); axes[0].set_ylabel("count")
axes[1].hist(vh_all, bins=80)
axes[1].set_title("VH (dB) — sampled pixels")
axes[1].set_xlabel("dB")
axes[2].hist(chip_pct.dropna().values, bins=40)
axes[2].set_title("Per-chip flood % (of valid px, 446 chips)")
axes[2].set_xlabel("flood %"); axes[2].set_ylabel("n chips")
fig.tight_layout()
plt.show()
del vv_pool, vh_pool, vv_all, vh_all

## 11. Dataset-level statistics
Concise summary tables. **Not** normalization statistics — no global mean/std is computed for model scaling here (that belongs to NB02/preprocessing).

In [ ]:
# S1 value summary (from §5b subset — explicitly a sample, not a norm recipe)
display(s1_stats.set_index("band"))

# Label summary (full 446-chip census from §6)
display(label_totals.set_index("class"))

# Split summary: n chips + mean/median flood % per official split
split_flood = {}
for split, names in split_s1.items():
    vals = np.array([per_chip_flood_pct[n] for n in names], dtype=float)
    vals = vals[~np.isnan(vals)]
    split_flood[split] = {"n_chips": len(names),
                            "mean_flood_pct": round(float(vals.mean()), 3) if vals.size else np.nan,
                            "median_flood_pct": round(float(np.median(vals)), 3) if vals.size else np.nan,
                            "std_flood_pct": round(float(vals.std()), 3) if vals.size else np.nan,
                            "min_flood_pct": round(float(vals.min()), 3) if vals.size else np.nan,
                            "max_flood_pct": round(float(vals.max()), 3) if vals.size else np.nan}
split_stats = pd.DataFrame(split_flood).T[["n_chips", "mean_flood_pct", "median_flood_pct", "std_flood_pct", "min_flood_pct", "max_flood_pct"]]
display(split_stats)
print("Note: S1 rows are subset estimates; label/split rows are full-census.")

## 12. Data-quality checks
Explicit automated gates. Float comparisons use tolerances — never exact equality on transforms/bounds. Failures print diagnostics.

In [ ]:
checks = []

def check(name, cond, detail=""):
    checks.append({"check": name, "status": "PASS" if bool(cond) else "FAIL", "detail": detail})

# --- counts / splits ---
check("S1 count == 446", len(s1_files) == 446, f"found={len(s1_files)}")
check("label count == 446", len(lb_files) == 446, f"found={len(lb_files)}")
for s, e in EXPECTED_SPLITS.items():
    check(f"split {s} == {e}", counts.get(s) == e, f"found={counts.get(s)}")
check("split total == 446", total_ref == 446, f"found={total_ref}")
check("no split overlap", all(len(sets[a] & sets[b]) == 0 for i, a in enumerate(keys) for b in keys[i+1:]), "pairwise disjoint")
check("no missing split references", len(missing) == 0, f"missing={len(missing)}")
check("no unreferenced S1 chips", len(unref) == 0, f"unreferenced={len(unref)}" + (f" e.g.{unref[:2]}" if unref else ""))
check("every S1 has matching label", len(bad_map) == 0, f"mismatches={len(bad_map)}")

# --- S1 integrity ---
check("S1 all 2-band", bool((mdf['bands'] == 2).all()), str(mdf['bands'].unique()))
check("S1 shape == (512,512)", bool(((mdf['h'] == 512) & (mdf['w'] == 512)).all()),
      f"bad={(mdf['h'] != 512).sum() + (mdf['w'] != 512).sum()}")
check("S1 CRS is EPSG:4326", bool((mdf['crs'] == 'EPSG:4326').all()), str(mdf['crs'].unique()))
check("S1 dtype float32", bool(mdf['dtype'].apply(lambda d: tuple(d) == ('float32', 'float32')).all()), str(mdf['dtype'].unique()))
check("S1 resolution consistent", mdf[["res_x", "res_y"]].drop_duplicates().shape[0] == 1,
      str(mdf[["res_x", "res_y"]].drop_duplicates().values.tolist()))

# --- label integrity ---
check("label shape == (512,512)", bool(((ldf['h'] == 512) & (ldf['w'] == 512)).all()), "all 512x512" if ((ldf['h']==512)&(ldf['w']==512)).all() else "MISMATCH")
check("label CRS is EPSG:4326", bool((ldf['crs'] == 'EPSG:4326').all()), str(ldf['crs'].unique()))
check("label values subset of {-1,0,1}", tot_other == 0, f"other_px={tot_other} {dict(other_vals) if other_vals else ''}")

# --- S1/label geospatial alignment (tolerant float compare on a sample; full 446 pre-verified) ---
align_ok, align_bad = True, []
for p in s1_files[::50]:  # ~9 chips spread across events
    q = LABEL_DIR / p.name.replace("_S1Hand.tif", "_LabelHand.tif")
    with rasterio.open(p) as a, rasterio.open(q) as b:
        same = (a.crs == b.crs and a.shape == b.shape
                  and np.allclose(a.transform.to_affine().to_gdal() if hasattr(a.transform, 'to_affine') else tuple(a.transform),
                                    b.transform.to_affine().to_gdal() if hasattr(b.transform, 'to_affine') else tuple(b.transform),
                                    rtol=1e-6, atol=1e-9)
                  and np.allclose(a.bounds, b.bounds, rtol=1e-6, atol=1e-9))
        if not same:
            align_ok = False
            align_bad.append(p.name)
check("S1/label CRS+shape+transform+bounds agree (sampled, tolerant)", align_ok,
      f"sampled={len(s1_files[::50])} bad={align_bad[:3] if align_bad else []}")

# --- DEM / IMERG readability ---
check("DEM main tiles readable", len(dem_bad) == 0 and len(ddf) > 0,
      f"tiles={len(ddf)} unreadable={len(dem_bad)}")
check("DEM CRS EPSG:4326", bool((ddf['crs'] == 'EPSG:4326').all()), str(ddf['crs'].unique()))
check("IMERG 36 files, unique dates, all have precipitation",
      len(idf) == 36 and idf['date'].nunique() == 36 and bool(idf['has_precipitation'].all()),
      f"files={len(idf)} uniq_dates={idf['date'].nunique()} all_precip={bool(idf['has_precipitation'].all())}")
check("IMERG global grid 1x1800x3600", bool(((idf['dim_time']==1)&(idf['dim_lat']==1800)&(idf['dim_lon']==3600)).all()),
      str(idf[['dim_time','dim_lat','dim_lon']].drop_duplicates().values.tolist()))
check("IMERG all files opened without error", len(open_errors) == 0, f"errors={len(open_errors)}")

cdf = pd.DataFrame(checks)
display(cdf)
n_fail = int((cdf.status == 'FAIL').sum())
print(f"\nQUALITY GATES: {len(cdf)-n_fail}/{len(cdf)} PASS" + (" — ALL PASS" if n_fail == 0 else f" — {n_fail} FAIL (see rows above)"))

## 13. Final audit summary

In [ ]:
print("=" * 60)
print("DATASET AUDIT COMPLETE")
print("=" * 60)
print("Sen1Floods11: 446 hand-labeled S1 chips")
print(f"Splits — Train: {counts.get('train')} | Validation: {counts.get('valid')} | "
      f"Test: {counts.get('test')} | Bolivia holdout: {counts.get('bolivia')}")
print("Modalities available:")
print("  - Sentinel-1 VV/VH (512x512, float32 dB, EPSG:4326, NaN=NoData)")
print(f"  - Copernicus DEM ({len(ddf)} main tiles, EPSG:4326, ~30m, all readable)")
print(f"  - NASA IMERG precipitation ({len(idf)} daily files, 0.1-deg global grid)")
print(f"Flood prevalence (valid px, 446 chips): {100*tot_flood/valid_px:.3f}% | NoData px: {100*tot_nodata/total_px:.3f}% of all")
print("\nPASS:")
for _, r in cdf[cdf.status=='PASS'].iterrows():
    print(f"  [PASS] {r['check']}")
fails = cdf[cdf.status=='FAIL']
print("\nWARNINGS / FAILURES:" if len(fails) else "\nWarnings: none")
for _, r in fails.iterrows():
    print(f"  [FAIL] {r['check']} — {r['detail']}")
print("\nReady for NB02 (preprocessing/mosaicking/sampling) only if all gates PASS.")
print("Raw files untouched: this notebook performed reads only, wrote no datasets.")